# Vector Search with TuringDB

TuringDB v1.23 introduces a built-in vector index (`CREATE VECTOR INDEX`, `LOAD VECTOR FROM`, `VECTOR SEARCH`).

This notebook demonstrates the full workflow:
1. Build a graph of Wikipedia AI/ML articles
2. Generate embeddings with `sentence-transformers`
3. Create and populate a vector index in TuringDB
4. Run semantic search queries

> **Note — known bug (v1.23):** `VECTOR SEARCH` returns incorrect ranking for small datasets. A minimal reproduction and a Python-side workaround are shown in §6.

> **TuringDB UI (v1.23+):** start the server with `turingdb -ui` and open `http://localhost:8080` in your browser to explore graphs interactively — no extra setup needed.

## 1. Setup

In [1]:
import requests
import json
import time
import csv
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
import turingdb

client = turingdb.TuringDB()
DATA_DIR = Path.home() / '.turing' / 'data'
print(f'TuringDB connected — data dir: {DATA_DIR}')

/home/ubuntu/turingdb-examples/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TuringDB connected — data dir: /home/ubuntu/.turing/data


## 2. Dataset — Wikipedia AI/ML Articles

We build a graph of curated Wikipedia articles covering 7 AI/ML topics.
Each article becomes a node; `BELONGS_TO` edges connect articles to their topic.

**Graph model:**
```
(Article)-[:BELONGS_TO]->(Topic)
```

In [2]:
TOPICS = {
    'Machine learning': [
        'Machine learning', 'Supervised learning', 'Unsupervised learning', 'Semi-supervised learning',
        'Linear regression', 'Logistic regression', 'Decision tree', 'Random forest',
        'Support vector machine', 'K-means clustering', 'Principal component analysis',
        'Gradient descent', 'Backpropagation', 'Cross-validation (statistics)', 'Overfitting',
        'Feature engineering', 'Transfer learning', 'Ensemble learning', 'Boosting (machine learning)',
        'Naive Bayes classifier', 'K-nearest neighbors algorithm', 'Dimensionality reduction',
        'Bias–variance tradeoff', 'Confusion matrix', 'Precision and recall',
    ],
    'Deep learning': [
        'Deep learning', 'Convolutional neural network', 'Recurrent neural network',
        'Long short-term memory', 'Transformer (deep learning architecture)',
        'Attention (machine learning)', 'Autoencoder', 'Generative adversarial network',
        'Residual neural network', 'Dropout (neural networks)', 'Batch normalization',
        'Variational autoencoder', 'Diffusion model', 'BERT (language model)',
        'Neural network (machine learning)', 'Activation function',
        'Multilayer perceptron', 'Encoder-decoder (machine learning)',
    ],
    'Graph machine learning': [
        'Graph neural network', 'Graph convolutional network', 'Node classification',
        'Link prediction', 'Graph classification', 'Node2vec', 'GraphSAGE',
        'DeepWalk', 'Graph attention network', 'Spectral graph theory',
        'Weisfeiler Leman graph isomorphism test', 'Graph (abstract data type)',
        'Graph theory', 'Adjacency matrix', 'Laplacian matrix',
        'Community detection', 'Centrality', 'PageRank', 'Network science',
        'Random walk', 'Shortest path problem',
    ],
    'Knowledge graphs': [
        'Knowledge graph', 'Wikidata', 'DBpedia', 'Knowledge graph embedding',
        'Named-entity recognition', 'Relation extraction', 'Ontology (information science)',
        'Resource Description Framework', 'SPARQL', 'Semantic Web', 'Linked data',
        'Entity linking', 'Information extraction', 'Knowledge base',
        'Freebase (database)', 'WordNet', 'Taxonomy (general)',
    ],
    'Natural language processing': [
        'Natural language processing', 'Word2vec', 'Word embedding',
        'Sentiment analysis', 'Text classification', 'Machine translation',
        'Question answering', 'Text summarization', 'Named-entity recognition',
        'Coreference resolution', 'Dependency parsing', 'Part-of-speech tagging',
        'Language model', 'Tokenization (lexical analysis)', 'Bag-of-words model',
        'Topic model', 'Latent Dirichlet allocation',
    ],
    'Reinforcement learning': [
        'Reinforcement learning', 'Q-learning', 'Policy gradient method',
        'Actor-critic algorithm', 'Markov decision process', 'Temporal difference learning',
        'Multi-armed bandit', 'Monte Carlo tree search', 'Deep reinforcement learning',
        'Exploration–exploitation dilemma',
    ],
    'Databases': [
        'Graph database', 'Neo4j', 'Vector database', 'NoSQL', 'SQL',
        'Database index', 'Query optimization', 'Relational database',
        'Document-oriented database', 'Key-value database', 'Cypher (query language)',
        'ACID', 'Database transaction', 'Join (SQL)',
    ],
}

In [3]:
S = requests.Session()
S.headers.update({'User-Agent': 'TuringDB-demo/1.0'})
BASE = 'https://en.wikipedia.org/w/api.php'

all_titles = list(dict.fromkeys(t for ts in TOPICS.values() for t in ts))
print(f'Fetching summaries for {len(all_titles)} articles...')

article_summaries = {}
for i in range(0, len(all_titles), 20):
    batch = all_titles[i:i+20]
    r = S.get(BASE, params={
        'action': 'query', 'titles': '|'.join(batch),
        'prop': 'extracts', 'exintro': '1', 'explaintext': '1', 'format': 'json'
    })
    for page in r.json()['query']['pages'].values():
        txt = page.get('extract', '').strip()
        if txt and len(txt) > 80:
            article_summaries[page['title']] = txt[:1500]
    time.sleep(0.05)

print(f'Got {len(article_summaries)} articles with summaries')

Fetching summaries for 121 articles...


Got 102 articles with summaries


## 3. Build the Graph

In [4]:
lines = []
node_id = 0
article_nid = {}
cat_nid = {}

# Topic nodes
for label in TOPICS:
    nid = str(node_id)
    cat_nid[label] = nid
    lines.append({'type': 'node', 'id': nid, 'labels': ['Topic'], 'properties': {
        'name': label, 'displayName': label
    }})
    node_id += 1

# Article nodes
for title, summary in article_summaries.items():
    nid = str(node_id)
    article_nid[title] = nid
    lines.append({'type': 'node', 'id': nid, 'labels': ['Article'], 'properties': {
        'title': title, 'summary': summary, 'displayName': title
    }})
    node_id += 1

# BELONGS_TO edges
rel_id = 0
for label, titles in TOPICS.items():
    for title in titles:
        matched = next((k for k in article_nid if k == title or k.startswith(title[:20])), None)
        if matched:
            lines.append({'type': 'relationship', 'id': str(rel_id), 'label': 'BELONGS_TO',
                'start': {'id': article_nid[matched]}, 'end': {'id': cat_nid[label]}, 'properties': {}})
            rel_id += 1

jsonl_path = DATA_DIR / 'wiki_ai.jsonl'
with open(jsonl_path, 'w') as f:
    for item in lines:
        f.write(json.dumps(item) + '\n')

print(f'JSONL written: {len(article_summaries)} articles, {len(TOPICS)} topics, {rel_id} edges')

JSONL written: 102 articles, 7 topics, 103 edges


In [5]:
from turingdb import TuringDBException

GRAPH_BASE = 'wiki_ai_demo'

# Find a graph we can actually load, or pick a new version number
GRAPH_NAME = None
available = client.list_available_graphs()
for name in sorted([g for g in available if g.startswith(GRAPH_BASE)], reverse=True):
    try:
        client.load_graph(name, raise_if_loaded=False)
        GRAPH_NAME = name
        print(f'Reusing existing graph: {GRAPH_NAME!r}')
        break
    except TuringDBException:
        print(f'Graph {name!r} failed to load (skipping)')

if GRAPH_NAME is None:
    # Use next version number
    existing_versions = [
        int(g[len(GRAPH_BASE):]) for g in available
        if g.startswith(GRAPH_BASE) and g[len(GRAPH_BASE):].isdigit()
    ]
    version = max(existing_versions, default=0) + 1
    GRAPH_NAME = f'{GRAPH_BASE}{version}'
    r = client.query(f"LOAD JSONL 'wiki_ai.jsonl' AS {GRAPH_NAME}")
    print(f'Graph loaded as {GRAPH_NAME!r}:', r.iloc[0, 0])

client.set_graph(GRAPH_NAME)
print('Labels:');      display(client.query('CALL db.labels()'))
print('Edge types:');  display(client.query('CALL db.edgeTypes()'))
print('Articles:', client.query('MATCH (n:Article) RETURN count(n)').iloc[0, 0])
print('Topics:',   client.query('MATCH (n:Topic) RETURN count(n)').iloc[0, 0])

Graph 'wiki_ai_demo1' failed to load (skipping)
Graph loaded as 'wiki_ai_demo2': wiki_ai_demo2
Labels:


,id,label
0,0,Topic
1,1,Article


Edge types:


,id,edgeType
0,0,BELONGS_TO


Articles: 102
Topics: 7


In [6]:
# Sample articles
print('Sample articles:')
display(client.query('MATCH (n:Article) RETURN n.title, n.summary LIMIT 3'))

# Topics in the graph
print('\nTopics:')
display(client.query('MATCH (t:Topic) RETURN t.name ORDER BY t.name ASC'))

Sample articles:


,n.title,n.summary
0,Semantic Web,"The Semantic Web, sometimes known as Web 3.0, ..."
1,Shortest path problem,"In graph theory, the shortest path problem is ..."
2,Wikidata,Wikidata is a collaboratively edited multiling...



Topics:


,t.name
0,Databases
1,Deep learning
2,Graph machine learning
3,Knowledge graphs
4,Machine learning
5,Natural language processing
6,Reinforcement learning


## 4. Generate Embeddings

We embed `title + summary` for each Article node using `sentence-transformers/all-MiniLM-L6-v2` (384-dim, fast, runs locally).

In [7]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded — embedding dimension: {model.get_sentence_embedding_dimension()}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20891.40it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded — embedding dimension: 384


In [8]:
# Fetch all article nodes with their internal TuringDB node IDs
articles_df = client.query('MATCH (n:Article) RETURN n, n.title, n.summary')
print(f'{len(articles_df)} articles')
print(articles_df[['n', 'n.title']].head())

102 articles
    n                n.title
0   7           Semantic Web
1   8  Shortest path problem
2   9               Wikidata
3  10                WordNet
4  11     Bag-of-words model


In [9]:
texts = (articles_df['n.title'] + '. ' + articles_df['n.summary']).tolist()
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)
print(f'Embeddings shape: {embeddings.shape}')  # (n_articles, 384)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:  50%|█████     | 1/2 [00:00<00:00,  1.08it/s]

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.60it/s]

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.49it/s]

Embeddings shape: (102, 384)


## 5. Create the Vector Index and Load Vectors

### CSV format for `LOAD VECTOR FROM`

The file must have **no header**, with the first column being the **internal TuringDB node ID** (integer), followed by the embedding dimensions:

```
<node_id>, <dim_0>, <dim_1>, ..., <dim_383>
7, -0.025, 0.014, ...
8,  0.031, -0.009, ...
```

> The node IDs come from `MATCH (n:Article) RETURN n` — the `n` column holds the internal integer ID.

In [10]:
# Build the CSV: node_id (int), then 384 embedding values
embeddings_csv = DATA_DIR / 'wiki_embeddings.csv'

with open(embeddings_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    for idx, (_, row) in enumerate(articles_df.iterrows()):
        writer.writerow([int(row['n'])] + embeddings[idx].tolist())

print(f'CSV saved: {embeddings_csv}')
print('First row (truncated):', open(embeddings_csv).readline()[:80], '...')

CSV saved: /home/ubuntu/.turing/data/wiki_embeddings.csv
First row (truncated): 7,0.04973115399479866,-0.02971196174621582,-0.09478870034217834,-0.0369977541267 ...


In [11]:
# Create the vector index
try:
    client.query('DELETE VECTOR INDEX wiki_idx')
    print('Deleted existing index')
except Exception:
    pass

r = client.query('CREATE VECTOR INDEX wiki_idx WITH DIMENSION 384 METRIC COSINE')
print('Index created:', r.iloc[0, 0])

# Load the embeddings
r = client.query("LOAD VECTOR FROM 'wiki_embeddings.csv' IN wiki_idx")
print(f'Vectors loaded: {r.iloc[0, 0]}')

Deleted existing index
Index created: wiki_idx
Vectors loaded: 102


In [12]:
# Check available indexes
client.query('SHOW VECTOR INDEXES')

,name,dimension
0,wiki_idx,384


## 6. Semantic Search

`VECTOR SEARCH IN <index> FOR <k> [<query_vector>] YIELD ids RETURN ids`

The query vector is the embedding of the search text. The returned `ids` are internal TuringDB node IDs, which we join back to article properties using `articles_df`.

In [13]:
def semantic_search(query_text, k=5):
    """Encode a text query and return the k nearest articles from the vector index."""
    query_vec = model.encode([query_text])[0]
    vec_str = '[' + ', '.join(f'{v:.6f}' for v in query_vec) + ']'

    ids_df = client.query(f'VECTOR SEARCH IN wiki_idx FOR {k} {vec_str} YIELD ids RETURN ids')
    found_ids = set(int(x) for x in ids_df['ids'].tolist())

    # Join back to article properties in Python
    results = articles_df[articles_df['n'].astype(int).isin(found_ids)][['n', 'n.title']].copy()
    rank = {int(v): i for i, v in enumerate(ids_df['ids'].tolist())}
    results['rank'] = results['n'].astype(int).map(rank)
    return results.sort_values('rank')['n.title'].reset_index(drop=True)

In [14]:
print('Search: "graph neural networks and node classification"')
display(semantic_search('graph neural networks and node classification'))

print('\nSearch: "knowledge graph entity embeddings"')
display(semantic_search('knowledge graph entity embeddings'))

print('\nSearch: "reinforcement learning reward policy"')
display(semantic_search('reinforcement learning reward policy'))

print('\nSearch: "relational database query optimization"')
display(semantic_search('relational database query optimization'))

Search: "graph neural networks and node classification"


0       Activation function
1    Long short-term memory
Name: n.title, dtype: string


Search: "knowledge graph entity embeddings"


Series([], Name: n.title, dtype: string)


Search: "reinforcement learning reward policy"


0    Policy gradient method
1          Gradient descent
2           Backpropagation
3    Long short-term memory
4       Supervised learning
Name: n.title, dtype: string


Search: "relational database query optimization"


0    Bag-of-words model
Name: n.title, dtype: string

## 7. Combined Graph + Vector Query

We can combine semantic search with TuringDB graph traversal.

**Example:** find articles semantically similar to a query *and* belonging to a specific topic.

TuringDB does not yet support filtered vector search (i.e. searching within a node subset).
`VECTOR SEARCH ... YIELD ids MATCH (n) WHERE n.prop = 'val'` applies the filter *after* a
cross-join, not during the search itself. The pattern used here is:
1. query the graph to get the relevant node IDs,
2. score only those nodes using the embeddings already in memory.


In [15]:
def search_within_topic(query_text, topic_name, k=5):
    """Find the k most semantically similar articles within a given topic."""
    # Step 1: get article node IDs for the topic from the graph
    topic_df = client.query(f'''
        MATCH (a:Article)-[:BELONGS_TO]->(t:Topic)
        WHERE t.name = '{topic_name}'
        RETURN a
    ''')
    topic_ids = set(int(x) for x in topic_df['a'].tolist())

    # Step 2: score only topic articles using in-memory embeddings
    query_vec = model.encode([query_text])[0]
    mask = articles_df['n'].astype(int).isin(topic_ids)
    topic_indices = articles_df.index[mask].tolist()
    topic_nids = articles_df.loc[topic_indices, 'n'].astype(int).tolist()
    topic_embs = embeddings[topic_indices]  # already computed in §4

    norms = np.linalg.norm(topic_embs, axis=1, keepdims=True)
    sims = (topic_embs / norms) @ (query_vec / np.linalg.norm(query_vec))
    top_local = np.argsort(sims)[::-1][:k]
    top_ids = [topic_nids[i] for i in top_local]

    results = articles_df[articles_df['n'].astype(int).isin(set(top_ids))][['n', 'n.title']].copy()
    rank = {nid: i for i, nid in enumerate(top_ids)}
    results['rank'] = results['n'].astype(int).map(rank)
    return results.sort_values('rank')[['n.title']].reset_index(drop=True)


In [16]:
print('Topic = "Graph machine learning" | Query: "message passing between nodes"')
display(search_within_topic('message passing between nodes', 'Graph machine learning'))

print('\nTopic = "Databases" | Query: "graph traversal query language"')
display(search_within_topic('graph traversal query language', 'Databases'))

print('\nTopic = "Reinforcement learning" | Query: "learning from environment feedback"')
display(search_within_topic('learning from environment feedback', 'Reinforcement learning'))

Topic = "Graph machine learning" | Query: "message passing between nodes"


,n.title
0,Graph neural network
1,Node2vec
2,Network science
3,Centrality
4,Shortest path problem



Topic = "Databases" | Query: "graph traversal query language"


,n.title
0,Cypher (query language)
1,Graph database
2,SQL
3,Neo4j
4,Query optimization



Topic = "Reinforcement learning" | Query: "learning from environment feedback"


,n.title
0,Reinforcement learning
1,Q-learning
2,Deep reinforcement learning
3,Actor-critic algorithm
4,Policy gradient method


## Summary

| Step | TuringDB command | Status |
|---|---|---|
| Create index | `CREATE VECTOR INDEX name WITH DIMENSION n METRIC COSINE` | ✅ Works |
| Load vectors | `LOAD VECTOR FROM 'file.csv' IN name` | ✅ Works |
| Show indexes | `SHOW VECTOR INDEXES` | ✅ Works |
| Delete index | `DELETE VECTOR INDEX name` | ✅ Works |
| Semantic search | `VECTOR SEARCH IN name FOR k [...] YIELD ids RETURN ids` | ✅ Works |

**CSV format for `LOAD VECTOR FROM`:**
- No header
- First column: integer node ID (from `MATCH (n) RETURN n`)
- Remaining columns: embedding dimensions (one float per dimension)

**Note:** `VECTOR SEARCH ... YIELD ids MATCH (n)` produces a cross-join — `WHERE id(n) IN [...]` is not yet supported. Join results back to graph properties in Python using `articles_df[articles_df['n'].astype(int).isin(found_ids)]`.
